# あみたろ ITAコーパス ファインチューニング (Piper-Plus)

6言語マルチリンガルベースモデル (571話者, 75 epoch) から、あみたろの声素材工房 ITAコーパス音声を使った日本語ファインチューニングを行います。

**ベースモデル:** `ayousanz/piper-plus-base` (6lang, epoch=74-step=504712)

**データセット:** あみたろ ITAコーパス (424文 x 4スタイル = ~1,696発話)

**ライセンス:** 商用利用OK (クレジット必須)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ayutaz/piper-plus/blob/dev/notebooks/amitaro_finetune.ipynb)

## 0. GPU 確認

In [ ]:
!nvidia-smi
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

## 1. 環境セットアップ

In [ ]:
# piper-plus リポジトリをクローン
!git clone https://github.com/ayutaz/piper-plus.git /content/piper-plus
%cd /content/piper-plus

In [ ]:
# uv インストール + 依存関係セットアップ
!pip install uv
!uv sync

In [ ]:
# 追加依存 (音声処理用)
!uv pip install soxr soundfile

## 2. ベースモデル (6lang チェックポイント) ダウンロード

In [ ]:
!pip install huggingface_hub

In [ ]:
from huggingface_hub import hf_hub_download
import os

BASE_MODEL_REPO = "ayousanz/piper-plus-base"
CHECKPOINT_FILENAME = "epoch=74-step=504712.ckpt"

os.makedirs("/content/base-model", exist_ok=True)

ckpt_path = hf_hub_download(
    repo_id=BASE_MODEL_REPO,
    filename=CHECKPOINT_FILENAME,
    local_dir="/content/base-model",
)
print(f"Checkpoint downloaded: {ckpt_path}")

BASE_CHECKPOINT = f"/content/base-model/{CHECKPOINT_FILENAME}"
print(f"Size: {os.path.getsize(BASE_CHECKPOINT) / 1024**2:.1f} MB")

## 3. あみたろ ITAコーパス ダウンロード & 展開

あみたろの声素材工房からITAコーパス音声をダウンロードします。

**手動ダウンロードが必要な場合:**
1. https://amitaro.net/voice/corpus-list/ita/ からダウンロード
2. Colab にアップロード

以下のセルでは Google Drive 経由での読み込みにも対応しています。

In [ ]:
# === 設定 ===
# あみたろ音声の配置方法を選択してください:
#   "upload"  : このセル実行後に手動アップロード
#   "gdrive"  : Google Drive からコピー
SOURCE_METHOD = "upload"  # "upload" or "gdrive"

# Google Drive 使用時のパス (SOURCE_METHOD="gdrive" の場合)
GDRIVE_ZIP_PATH = "/content/drive/MyDrive/amitaro_ita.zip"

# 展開先
AMITARO_RAW_DIR = "/content/amitaro-raw"

In [ ]:
import os
import zipfile
import glob

os.makedirs(AMITARO_RAW_DIR, exist_ok=True)

if SOURCE_METHOD == "gdrive":
    from google.colab import drive
    drive.mount("/content/drive")
    
    if GDRIVE_ZIP_PATH.endswith(".zip"):
        with zipfile.ZipFile(GDRIVE_ZIP_PATH, "r") as zf:
            zf.extractall(AMITARO_RAW_DIR)
        print(f"Extracted to {AMITARO_RAW_DIR}")
    else:
        import shutil
        shutil.copytree(GDRIVE_ZIP_PATH, AMITARO_RAW_DIR, dirs_exist_ok=True)
        print(f"Copied to {AMITARO_RAW_DIR}")

elif SOURCE_METHOD == "upload":
    from google.colab import files
    print("ZIP ファイルをアップロードしてください (あみたろ ITA コーパス):")
    uploaded = files.upload()
    for fname in uploaded:
        if fname.endswith(".zip"):
            with zipfile.ZipFile(fname, "r") as zf:
                zf.extractall(AMITARO_RAW_DIR)
            print(f"Extracted {fname} to {AMITARO_RAW_DIR}")

# 展開結果を確認
wav_files = glob.glob(f"{AMITARO_RAW_DIR}/**/*.wav", recursive=True)
print(f"\nFound {len(wav_files)} WAV files")
if wav_files:
    print(f"Example: {wav_files[0]}")

## 4. 音声前処理 & LJSpeech 形式変換

あみたろ音声を以下の手順で処理します:
1. 22,050Hz にリサンプリング
2. 音量正規化
3. LJSpeech 形式 (wavs/ + metadata.csv) に変換

In [ ]:
import os
import csv
import glob
import re
import numpy as np
import soundfile as sf

try:
    import soxr
    HAS_SOXR = True
except ImportError:
    import torchaudio
    HAS_SOXR = False
    print("soxr not available, using torchaudio for resampling")

TARGET_SR = 22050
LJSPEECH_DIR = "/content/amitaro-ljspeech"
WAVS_DIR = os.path.join(LJSPEECH_DIR, "wavs")
os.makedirs(WAVS_DIR, exist_ok=True)


def resample_and_normalize(audio_path, output_path, target_sr=TARGET_SR):
    """Resample to target_sr and peak-normalize."""
    audio, src_sr = sf.read(audio_path, dtype="float32")
    if audio.ndim > 1:
        audio = audio.mean(axis=1)  # stereo -> mono
    
    # Resample
    if src_sr != target_sr:
        if HAS_SOXR:
            audio = soxr.resample(audio, src_sr, target_sr, quality="HQ")
        else:
            import torch
            resampler = torchaudio.transforms.Resample(src_sr, target_sr)
            audio = resampler(torch.from_numpy(audio).unsqueeze(0)).squeeze(0).numpy()
    
    # Peak normalize
    peak = np.abs(audio).max()
    if peak > 0:
        audio = audio / peak * 0.95
    
    sf.write(output_path, audio, target_sr, subtype="PCM_16")
    return len(audio) / target_sr  # duration in seconds


# あみたろ ITA コーパスの構造を自動検出
# 一般的な構造: RECITATION/ と EMOTION/ 以下に WAV + テキスト
wav_files = sorted(glob.glob(f"{AMITARO_RAW_DIR}/**/*.wav", recursive=True))
print(f"Total WAV files found: {len(wav_files)}")

# ディレクトリ構造を表示
dirs = set()
for f in wav_files:
    rel = os.path.relpath(os.path.dirname(f), AMITARO_RAW_DIR)
    dirs.add(rel)
print(f"Directories: {sorted(dirs)}")

In [ ]:
# === ITA コーパス テキスト読み込み ===
# ITA コーパスには RECITATION (朗読) と EMOTION (感情) の2カテゴリがあります。
# テキストは同梱の transcript ファイルか、ITA コーパス公式テキストから取得します。
#
# あみたろ配布の一般的な構造:
#   ノーマル/  RECITATION001.wav ~ RECITATION324.wav, EMOTION001.wav ~ EMOTION100.wav
#   るんるん/  同上
#   よふかし/ 同上
#   ぷんすか/ 同上
#
# ITA コーパスのテキストリスト (公式) を使用します。
# テキストファイルが同梱されている場合はそちらを優先します。

import json

def find_transcript_files(raw_dir):
    """Search for transcript/text files in the raw directory."""
    patterns = ["*.txt", "*.csv", "*.tsv", "*transcript*", "*text*"]
    found = []
    for pat in patterns:
        found.extend(glob.glob(f"{raw_dir}/**/{pat}", recursive=True))
    return sorted(set(found))

transcript_files = find_transcript_files(AMITARO_RAW_DIR)
print(f"Found transcript files: {transcript_files}")
print()
print("If no transcript files found, the ITA corpus text list will be downloaded in the next cell.")

In [ ]:
# === ITA コーパス公式テキストリスト ===
# ITA コーパスの RECITATION (324文) と EMOTION (100文) のテキストを取得します。
# 公式リポジトリ: https://github.com/mmorise/ita-corpus

ITA_CORPUS_DIR = "/content/ita-corpus"
!git clone https://github.com/mmorise/ita-corpus.git {ITA_CORPUS_DIR} 2>/dev/null || echo "Already cloned"

def load_ita_corpus_texts(ita_dir):
    """Load ITA corpus texts from the official repository."""
    texts = {}
    
    # RECITATION: recitation001 ~ recitation324
    recitation_dir = os.path.join(ita_dir, "recitation")
    if os.path.isdir(recitation_dir):
        for txt_file in sorted(glob.glob(f"{recitation_dir}/*.txt")):
            basename = os.path.splitext(os.path.basename(txt_file))[0]
            with open(txt_file, encoding="utf-8") as f:
                text = f.read().strip()
            # e.g. "recitation001" -> "RECITATION001"
            key = basename.upper()
            texts[key] = text
    
    # EMOTION: emotion001 ~ emotion100
    emotion_dir = os.path.join(ita_dir, "emotion")
    if os.path.isdir(emotion_dir):
        for txt_file in sorted(glob.glob(f"{emotion_dir}/*.txt")):
            basename = os.path.splitext(os.path.basename(txt_file))[0]
            with open(txt_file, encoding="utf-8") as f:
                text = f.read().strip()
            key = basename.upper()
            texts[key] = text
    
    print(f"Loaded {len(texts)} ITA corpus texts")
    print(f"  RECITATION: {sum(1 for k in texts if k.startswith('RECITATION'))}")
    print(f"  EMOTION: {sum(1 for k in texts if k.startswith('EMOTION'))}")
    return texts

ita_texts = load_ita_corpus_texts(ITA_CORPUS_DIR)

# サンプル表示
for key in list(ita_texts.keys())[:3]:
    print(f"  {key}: {ita_texts[key][:50]}...")

In [ ]:
# === LJSpeech 形式に変換 ===
# wavs/amitaro_STYLE_RECITATION001.wav + metadata.csv

import re
from collections import defaultdict

metadata = []  # [(wav_id, text), ...]
stats = defaultdict(int)
skipped = []

for wav_path in sorted(wav_files):
    basename = os.path.splitext(os.path.basename(wav_path))[0]  # e.g. "RECITATION001"
    
    # スタイル名をディレクトリから推定
    parent_dir = os.path.basename(os.path.dirname(wav_path))
    # 日本語ディレクトリ名を英語に変換
    style_map = {
        "ノーマル": "normal", "normal": "normal",
        "るんるん": "runrun", "runrun": "runrun",
        "よふかし": "yofukashi", "yofukashi": "yofukashi",
        "ぷんすか": "punsuka", "punsuka": "punsuka",
    }
    style = style_map.get(parent_dir, parent_dir.lower())
    
    # テキストキーを抽出 (e.g. "RECITATION001", "EMOTION001")
    text_key = basename.upper()
    # ファイル名に余分なプレフィックスがある場合に対応
    match = re.search(r"(RECITATION\d+|EMOTION\d+)", text_key)
    if match:
        text_key = match.group(1)
    
    text = ita_texts.get(text_key)
    if text is None:
        skipped.append((wav_path, text_key))
        continue
    
    # WAV ID: amitaro_{style}_{basename}
    wav_id = f"amitaro_{style}_{basename}"
    output_wav = os.path.join(WAVS_DIR, f"{wav_id}.wav")
    
    # リサンプリング & 正規化
    duration = resample_and_normalize(wav_path, output_wav)
    
    metadata.append((wav_id, text))
    stats[style] += 1

# metadata.csv 書き出し (LJSpeech 形式: id|text|text)
metadata_path = os.path.join(LJSPEECH_DIR, "metadata.csv")
with open(metadata_path, "w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f, delimiter="|")
    for wav_id, text in metadata:
        writer.writerow([wav_id, text, text])

print(f"\nLJSpeech dataset created at: {LJSPEECH_DIR}")
print(f"Total utterances: {len(metadata)}")
print(f"Styles: {dict(stats)}")
if skipped:
    print(f"Skipped (no text match): {len(skipped)}")
    for path, key in skipped[:5]:
        print(f"  {key}: {path}")

## 5. ファインチューニング用データセット作成

LJSpeech 形式のデータから、Piper 学習用の `dataset.jsonl` + `config.json` を生成します。

In [ ]:
import sys
sys.path.insert(0, "/content/piper-plus/src/python")
sys.path.insert(0, "/content/piper-plus/src/python/g2p")

import json
import csv
import os
import numpy as np
import torch
from pathlib import Path
from hashlib import sha256
from tqdm import tqdm

from piper_plus_g2p.encode.id_maps import get_phoneme_id_map
from piper_train.infer_onnx import text_to_phoneme_ids_and_prosody
from piper_train.vits.mel_processing import spectrogram_torch

# === 設定 ===
DATASET_DIR = "/content/dataset-amitaro-finetune-6lang"
CACHE_DIR = os.path.join(DATASET_DIR, "cache")
SAMPLE_RATE = 22050
LANGUAGE_KEY = "ja-en-zh-es-fr-pt"  # 6lang canonical key
LANGUAGE_ID = 0  # ja = 0

# Spectrogram parameters (must match training config)
FILTER_LENGTH = 1024
WINDOW_LENGTH = 1024
HOP_LENGTH = 256

os.makedirs(CACHE_DIR, exist_ok=True)

# ID map
id_map = get_phoneme_id_map(LANGUAGE_KEY)
num_symbols = max(max(ids) for ids in id_map.values()) + 1

# Language ID map (for text_to_phoneme_ids_and_prosody auto-promotion)
LANGUAGE_ID_MAP = {"ja": 0, "en": 1, "zh": 2, "es": 3, "fr": 4, "pt": 5}

print(f"Language key: {LANGUAGE_KEY}")
print(f"Num symbols: {num_symbols}")
print(f"ID map size: {len(id_map)} symbols")

In [ ]:
def encode_text(text, language="ja"):
    """Phonemize text and encode to phoneme IDs using the project's standard pipeline."""
    phoneme_ids, prosody_features = text_to_phoneme_ids_and_prosody(
        text,
        phoneme_id_map=id_map,
        language=language,
        language_id_map=LANGUAGE_ID_MAP,
    )
    return phoneme_ids, prosody_features


def cache_audio(wav_path, cache_dir, sample_rate=SAMPLE_RATE):
    """Cache normalized audio and spectrogram."""
    wav_path = Path(wav_path).absolute()
    cache_id = sha256(str(wav_path).encode()).hexdigest()
    norm_path = Path(cache_dir) / f"{cache_id}.pt"
    spec_path = Path(cache_dir) / f"{cache_id}.spec.pt"
    
    if not norm_path.exists():
        audio, sr = sf.read(str(wav_path), dtype="float32")
        if audio.ndim > 1:
            audio = audio.mean(axis=1)
        if sr != sample_rate:
            if HAS_SOXR:
                audio = soxr.resample(audio, sr, sample_rate, quality="HQ")
            else:
                import torchaudio
                resampler = torchaudio.transforms.Resample(sr, sample_rate)
                audio = resampler(torch.from_numpy(audio).unsqueeze(0)).squeeze(0).numpy()
        
        audio_tensor = torch.from_numpy(audio).unsqueeze(0)  # (1, samples)
        torch.save(audio_tensor, str(norm_path))
    else:
        audio_tensor = torch.load(str(norm_path), map_location="cpu", weights_only=False)
    
    if not spec_path.exists():
        spec = spectrogram_torch(
            audio_tensor,
            n_fft=FILTER_LENGTH,
            sampling_rate=sample_rate,
            hop_size=HOP_LENGTH,
            win_size=WINDOW_LENGTH,
            center=False,
        ).squeeze(0)
        # Save as float16 to save disk space
        torch.save(spec.half(), str(spec_path))
    
    return str(norm_path), str(spec_path)


# テスト
test_ids, test_prosody = encode_text("こんにちは、今日は良い天気ですね。")
print(f"Test IDs ({len(test_ids)}): {test_ids[:20]}...")
print(f"Prosody features: {len(test_prosody)} entries")

In [ ]:
# === dataset.jsonl 生成 ===
dataset_jsonl_path = os.path.join(DATASET_DIR, "dataset.jsonl")
utterances = []
errors = []

# metadata.csv を読み込み
with open(metadata_path, encoding="utf-8") as f:
    reader = csv.reader(f, delimiter="|")
    rows = list(reader)

print(f"Processing {len(rows)} utterances...")

for wav_id, text, _ in tqdm(rows):
    wav_path = os.path.join(WAVS_DIR, f"{wav_id}.wav")
    if not os.path.exists(wav_path):
        errors.append(f"Missing: {wav_path}")
        continue
    
    try:
        # Phonemize
        phoneme_ids, prosody_features = encode_text(text, language="ja")
        
        # Cache audio
        norm_path, spec_path = cache_audio(wav_path, CACHE_DIR)
        
        # Convert prosody_features to serializable format
        prosody_serializable = None
        if prosody_features:
            prosody_serializable = [
                {"a1": p["a1"], "a2": p["a2"], "a3": p["a3"]} if p else None
                for p in prosody_features
            ]
        
        utterance = {
            "text": text,
            "phoneme_ids": phoneme_ids,
            "audio_norm_path": norm_path,
            "audio_spec_path": spec_path,
            "speaker_id": 0,
            "language_id": LANGUAGE_ID,
        }
        if prosody_serializable:
            utterance["prosody_features"] = prosody_serializable
        
        utterances.append(utterance)
    except Exception as e:
        errors.append(f"Error processing {wav_id}: {e}")

# Write dataset.jsonl
with open(dataset_jsonl_path, "w", encoding="utf-8") as f:
    for utt in utterances:
        f.write(json.dumps(utt, ensure_ascii=False) + "\n")

print(f"\nDataset written: {dataset_jsonl_path}")
print(f"Total utterances: {len(utterances)}")
if errors:
    print(f"Errors: {len(errors)}")
    for e in errors[:5]:
        print(f"  {e}")

In [ ]:
# === config.json 生成 ===
# 6lang ベースモデルと同じ設定 (173 symbols, 6 languages)

LANGUAGE_ID_MAP = {"ja": 0, "en": 1, "zh": 2, "es": 3, "fr": 4, "pt": 5}

config = {
    "audio": {
        "sample_rate": SAMPLE_RATE,
        "filter_length": FILTER_LENGTH,
        "hop_length": HOP_LENGTH,
        "win_length": WINDOW_LENGTH,
    },
    "num_symbols": num_symbols,
    "num_speakers": 1,
    "num_languages": len(LANGUAGE_ID_MAP),
    "language_id_map": LANGUAGE_ID_MAP,
    "speaker_id_map": {"amitaro": 0},
    "phoneme_id_map": {k: v for k, v in id_map.items()},
}

config_path = os.path.join(DATASET_DIR, "config.json")
with open(config_path, "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

print(f"Config written: {config_path}")
print(f"  num_symbols: {config['num_symbols']}")
print(f"  num_speakers: {config['num_speakers']}")
print(f"  num_languages: {config['num_languages']}")
print(f"  languages: {list(LANGUAGE_ID_MAP.keys())}")

## 6. ファインチューニング実行

Template B (シングルスピーカー) に準拠してファインチューニングを実行します。

| パラメータ | 値 | 理由 |
|-----------|-----|------|
| `--base_lr` | 2e-5 | catastrophic forgetting 防止 (事前学習の 1/10) |
| `--batch-size` | 4 | 小データセット向け |
| `--max_epochs` | 500 | 十分な gradient steps 確保 |
| `--freeze-dp` | auto | `--resume-from-multispeaker-checkpoint` で自動有効化 |

In [ ]:
# === 学習パラメータ ===
OUTPUT_DIR = "/content/output-amitaro-finetune-6lang"
MAX_EPOCHS = 500
BATCH_SIZE = 4
LEARNING_RATE = 2e-5
CHECKPOINT_EPOCHS = 50
SAMPLES_PER_SPEAKER = 4

# WandB (オプション)
WANDB_API_KEY = ""  # WandB API キーを設定する場合はここに入力

if WANDB_API_KEY:
    os.environ["WANDB_API_KEY"] = WANDB_API_KEY
    print("WandB logging enabled")
else:
    os.environ["WANDB_MODE"] = "disabled"
    print("WandB logging disabled")

In [ ]:
# === ファインチューニング実行 ===
!cd /content/piper-plus && uv run python -m piper_train \
    --dataset-dir {DATASET_DIR} \
    --prosody-dim 16 \
    --accelerator gpu --devices 1 --precision 32-true \
    --max_epochs {MAX_EPOCHS} --batch-size {BATCH_SIZE} --samples-per-speaker {SAMPLES_PER_SPEAKER} \
    --checkpoint-epochs {CHECKPOINT_EPOCHS} --quality medium \
    --base_lr {LEARNING_RATE} --disable_auto_lr_scaling \
    --ema-decay 0.9995 \
    --max-phoneme-ids 400 \
    --no-wavlm \
    --val-every-n-epochs 50 \
    --audio-log-epochs 50 \
    --resume-from-multispeaker-checkpoint {BASE_CHECKPOINT} \
    --default_root_dir {OUTPUT_DIR}

## 7. ONNX エクスポート

In [ ]:
import glob
import re

# 最新のチェックポイントを検索 (epoch 番号でソート)
ckpt_pattern = f"{OUTPUT_DIR}/lightning_logs/version_*/checkpoints/epoch=*.ckpt"
checkpoints = glob.glob(ckpt_pattern)

def extract_epoch(path):
    """Extract epoch number from checkpoint filename for correct sorting."""
    m = re.search(r"epoch=(\d+)", path)
    return int(m.group(1)) if m else -1

checkpoints = sorted(checkpoints, key=extract_epoch)

if checkpoints:
    FINAL_CHECKPOINT = checkpoints[-1]
    print(f"Latest checkpoint: {FINAL_CHECKPOINT}")
else:
    print("No checkpoints found! Training may not have completed.")
    FINAL_CHECKPOINT = None

In [ ]:
ONNX_OUTPUT = f"{OUTPUT_DIR}/amitaro-6lang.onnx"

if FINAL_CHECKPOINT:
    !cd /content/piper-plus && CUDA_VISIBLE_DEVICES="" uv run python -m piper_train.export_onnx \
        {FINAL_CHECKPOINT} \
        {ONNX_OUTPUT}
    
    if os.path.exists(ONNX_OUTPUT):
        print(f"\nONNX model exported: {ONNX_OUTPUT}")
        print(f"Size: {os.path.getsize(ONNX_OUTPUT) / 1024**2:.1f} MB")
        
        # config.json をモデルと同じ場所にコピー
        import shutil
        onnx_config = ONNX_OUTPUT.replace(".onnx", ".onnx.json")
        shutil.copy2(config_path, onnx_config)
        print(f"Config copied: {onnx_config}")

## 8. 推論テスト

In [ ]:
test_texts = [
    ("こんにちは、あみたろです。今日はとても良い天気ですね。", "ja"),
    ("吾輩は猫である。名前はまだ無い。", "ja"),
    ("Hello, how are you today?", "en"),
]

if os.path.exists(ONNX_OUTPUT):
    for i, (text, lang) in enumerate(test_texts):
        # infer_onnx --text outputs to {output_dir}/output.wav (fixed name)
        # Use separate output dirs per test to avoid overwriting
        test_output_dir = f"/content/test_output_{i}_{lang}"
        os.makedirs(test_output_dir, exist_ok=True)
        !cd /content/piper-plus && CUDA_VISIBLE_DEVICES="" uv run python -m piper_train.infer_onnx \
            --model {ONNX_OUTPUT} \
            --config {config_path} \
            --output-dir {test_output_dir} \
            --text "{text}" \
            --language {LANGUAGE_KEY} --speaker-id 0 --noise-scale 0.667
        print(f"Generated: {text[:30]}...")
else:
    print("ONNX model not found. Export first.")

In [ ]:
# Colab で音声を再生
from IPython.display import Audio, display

for i, (text, lang) in enumerate(test_texts):
    wav_path = f"/content/test_output_{i}_{lang}/output.wav"
    if os.path.exists(wav_path):
        print(f"\n[{lang}] {text[:40]}...")
        display(Audio(wav_path))
    else:
        print(f"\nNot found: {wav_path}")

## 9. モデル保存 (Google Drive)

学習済みモデルを Google Drive に保存します。

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import shutil

SAVE_DIR = "/content/drive/MyDrive/piper-plus-models/amitaro-6lang"
os.makedirs(SAVE_DIR, exist_ok=True)

# ONNX モデル + config をコピー
if os.path.exists(ONNX_OUTPUT):
    shutil.copy2(ONNX_OUTPUT, SAVE_DIR)
    shutil.copy2(ONNX_OUTPUT.replace(".onnx", ".onnx.json"), SAVE_DIR)
    print(f"Model saved to: {SAVE_DIR}")

# 最終チェックポイントもバックアップ
if FINAL_CHECKPOINT:
    shutil.copy2(FINAL_CHECKPOINT, SAVE_DIR)
    print(f"Checkpoint saved to: {SAVE_DIR}")

!ls -lh {SAVE_DIR}

## 10. クリーンアップ

不要なキャッシュファイルを削除してディスク容量を確保します。

In [ ]:
# キャッシュディレクトリのサイズ確認
!du -sh {CACHE_DIR} 2>/dev/null
!df -h /content

# 必要に応じてキャッシュを削除
# !rm -rf {CACHE_DIR}